In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [8]:
import os
path = "/content/drive/MyDrive/BATTERY_SOC_PROJECT/SOC DATA SET"

In [9]:
import os
import json
import time
import hashlib
import joblib
from datetime import datetime

In [10]:
BASE_PATH = "/content/drive/MyDrive/BATTERY_SOC_PROJECT"
DAY22_PATH = os.path.join(BASE_PATH, "day22_outputs")
DAY23_PATH = os.path.join(BASE_PATH, "day23_outputs")
os.makedirs(DAY23_PATH, exist_ok=True)
SOC_MODEL_PATH = os.path.join(DAY22_PATH,"FINAL_SOC_XGBoost_day22.joblib")
REMAINING_MODEL_PATH = os.path.join(
    DAY22_PATH,"FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib")
FEATURE_PATH = os.path.join(DAY22_PATH,"day22_final_features.json")
SELECTION_PATH = os.path.join(DAY22_PATH,"day22_model_selection.json")
FREEZE_RECORD_PATH = os.path.join(DAY22_PATH,"DAY22_FREEZE_RECORD.json")
print("DAY 23 — FINAL VALIDATION & RUNTIME VERIFICATION")

DAY 23 — FINAL VALIDATION & RUNTIME VERIFICATION


In [11]:
required_files = [SOC_MODEL_PATH,REMAINING_MODEL_PATH,FEATURE_PATH,SELECTION_PATH,FREEZE_RECORD_PATH]
print("\n[1] Checking Day 22 frozen artifacts...")
missing_files = []
for file_path in required_files:
    if os.path.exists(file_path):
        print(" ", os.path.basename(file_path))
    else:
        print("✗ MISSING:", os.path.basename(file_path))
        missing_files.append(file_path)
if missing_files:
    raise FileNotFoundError(
        "One or more required Day 22 artifacts are missing.")
print("All required Day 22 artifacts are present.")


[1] Checking Day 22 frozen artifacts...
  FINAL_SOC_XGBoost_day22.joblib
  FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib
  day22_final_features.json
  day22_model_selection.json
  DAY22_FREEZE_RECORD.json
All required Day 22 artifacts are present.


In [13]:
print("\n Loading frozen final models...")
load_start = time.perf_counter()
soc_model = joblib.load(SOC_MODEL_PATH)
soc_load_time = time.perf_counter() - load_start
load_start = time.perf_counter()
remaining_model = joblib.load(REMAINING_MODEL_PATH)
remaining_load_time = time.perf_counter() - load_start
print("SOC XGBoost model loaded")
print("Remaining Charging Time Random Forest loaded")
print(f"  SOC model load time: {soc_load_time:.6f} seconds")
print(
    f"  Remaining Time model load time: "
    f"{remaining_load_time:.6f} seconds"
)


 Loading frozen final models...
SOC XGBoost model loaded
Remaining Charging Time Random Forest loaded
  SOC model load time: 0.009264 seconds
  Remaining Time model load time: 0.084661 seconds


In [15]:
print("\nVerifying frozen feature configuration...")
with open(FEATURE_PATH, "r") as f:
    final_features = json.load(f)
expected_features = ["Voltage","Current","Temperature","Elapsed Time","Cycle","WhAccu","Power","Capacity"]
if final_features == expected_features:
    print("Final feature configuration verified.")
else:
    print("Feature configuration differs from expected list.")
    print("Stored:", final_features)
print("\nFinal features:")
for i, feature in enumerate(final_features, start=1):
    print(f"  {i}. {feature}")


Verifying frozen feature configuration...
Final feature configuration verified.

Final features:
  1. Voltage
  2. Current
  3. Temperature
  4. Elapsed Time
  5. Cycle
  6. WhAccu
  7. Power
  8. Capacity


In [17]:
print("\nVerifying final model selection record...")
with open(SELECTION_PATH, "r") as f:
    selection = json.load(f)
soc_selection = selection["SOC_task"]
remaining_selection = selection["Remaining_Charging_Time_task"]
print(
    f"SOC final model: "
    f"{soc_selection['final_model']}"
)
print(
    f"Remaining Charging Time final model: "
    f"{remaining_selection['final_model']}"
)
print(
    f"SOC Test R² recorded: "
    f"{soc_selection['test_R2']}"
)
print(
    f"Remaining Time Test R² recorded: "
    f"{remaining_selection['test_R2']}"
)
print(
    f"Remaining Time Validation R² recorded: "
    f"{remaining_selection['validation_R2']}"
)


Verifying final model selection record...
SOC final model: XGBoost
Remaining Charging Time final model: Random Forest
SOC Test R² recorded: 0.990521
Remaining Time Test R² recorded: 0.998756
Remaining Time Validation R² recorded: 0.996502


In [19]:
print("\nVerifying freeze record...")
with open(FREEZE_RECORD_PATH, "r") as f:
    freeze_record = json.load(f)
print("Freeze status:", freeze_record.get("status"))
if freeze_record.get("status") == "FROZEN":
    print("Day 22 freeze record confirmed.")
else:
    print("Day 22 freeze record is not marked as FROZEN.")


Verifying freeze record...
Freeze status: FROZEN
Day 22 freeze record confirmed.


In [20]:
print("\nCreating artifact integrity hashes...")
def file_sha256(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)
    return sha256.hexdigest()
soc_hash = file_sha256(SOC_MODEL_PATH)
remaining_hash = file_sha256(REMAINING_MODEL_PATH)
print("SOC model SHA-256 generated")
print("Remaining Time model SHA-256 generated")


Creating artifact integrity hashes...
SOC model SHA-256 generated
Remaining Time model SHA-256 generated


In [22]:
print("\nRuntime readiness check...")
runtime_start = time.perf_counter()
soc_model_type = type(soc_model).__name__
remaining_model_type = type(remaining_model).__name__
runtime_check_time = time.perf_counter() - runtime_start
print("✓ SOC model object is accessible")
print("✓ Remaining Time model object is accessible")
print("  SOC object type:", soc_model_type)
print("  Remaining Time object type:", remaining_model_type)
print(f"  Runtime access check: {runtime_check_time:.6f} seconds")


Runtime readiness check...
✓ SOC model object is accessible
✓ Remaining Time model object is accessible
  SOC object type: dict
  Remaining Time object type: dict
  Runtime access check: 0.000099 seconds


In [23]:
validation_record = {
    "day": 23,
    "purpose": "Final validation and runtime verification",
    "date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "frozen_models_verified": {
        "SOC": {
            "model": "XGBoost",
            "artifact": "FINAL_SOC_XGBoost_day22.joblib",
            "test_R2": soc_selection["test_R2"],
            "load_time_seconds": round(soc_load_time, 6),
            "sha256": soc_hash
        },
        "Remaining_Charging_Time": {
            "model": "Random Forest",
            "artifact":
                "FINAL_Remaining_Charging_Time_Random_Forest_day22.joblib",
            "test_R2": remaining_selection["test_R2"],
            "validation_R2":
                remaining_selection["validation_R2"],
            "load_time_seconds":
                round(remaining_load_time, 6),
            "sha256": remaining_hash
        }
    },
    "final_features": final_features,
    "checks": {
        "all_day22_artifacts_present": True,
        "models_loaded_successfully": True,
        "feature_configuration_verified":
            final_features == expected_features,
       "freeze_status_verified":
    freeze_record.get("status") == "FROZEN",
        "retraining_performed": False,
        "new_prediction_analysis_performed": False,
        "repeated_error_analysis_performed": False
    },
    "status": "VALIDATED_FOR_NEXT_STAGE"
}
validation_file = os.path.join(
    DAY23_PATH,
    "DAY23_VALIDATION_RECORD.json"
)
with open(validation_file, "w") as f:
    json.dump(validation_record, f, indent=4)
print("Day 23 validation record saved")

Day 23 validation record saved


In [24]:
submission_text = f"""
DAY 23 — FINAL VALIDATION & RUNTIME VERIFICATION
The frozen Day 22 model artifacts were successfully verified and
loaded without retraining.
Final models:
1. SOC Prediction
   Model: XGBoost
   Test R²: {soc_selection["test_R2"]}
2. Remaining Charging Time Prediction
   Model: Random Forest
   Test R²: {remaining_selection["test_R2"]}
   Validation R²: {remaining_selection["validation_R2"]}
Final feature set:
{chr(10).join("- " + feature for feature in final_features)}
Artifact verification:
- Day 22 frozen model files present: YES
- SOC model loaded successfully: YES
- Remaining Time model loaded successfully: YES
- Feature configuration verified: YES
- Freeze record  verified: YES
- Model retraining: NO
- Repeated prediction/error analysis: NO
Runtime loading:
- SOC model load time: {soc_load_time:.6f} seconds
- Remaining Time model load time: {remaining_load_time:.6f} seconds
The frozen models are validated for the next project stage:
technical review, packaging and documentation.
"""
submission_file = os.path.join(DAY23_PATH,"DAY23_SUBMISSION_NOTE.txt")
with open(submission_file, "w") as f:
    f.write(submission_text.strip())
print("Day 23 submission note saved")

Day 23 submission note saved


In [26]:
print("\nCreated in day23_outputs:")
for file in sorted(os.listdir(DAY23_PATH)):
    print(" •", file)
print("\nFINAL STATUS: VALIDATED FOR NEXT STAGE")
print("No retraining performed.")
print("No repeated error analysis performed.")
print("Frozen Day 22 models verified successfully.")


Created in day23_outputs:
 • DAY23_SUBMISSION_NOTE.txt
 • DAY23_VALIDATION_RECORD.json

FINAL STATUS: VALIDATED FOR NEXT STAGE
No retraining performed.
No repeated error analysis performed.
Frozen Day 22 models verified successfully.
